In [ ]:
# Required Libraries
import pandas as pd
import numpy as np
import json
import yaml
from pathlib import Path
from typing import Dict, Any, Optional

print("✅ Libraries loaded")

## 1. Data Loader Class

In [ ]:
class DataLoader:
    """
    Universal data loader for common file formats.
    Supports: CSV, Excel, JSON, Parquet
    """
    
    SUPPORTED_FORMATS = ['.csv', '.xlsx', '.xls', '.json', '.parquet']
    
    @staticmethod
    def load(file_path: str, **kwargs) -> pd.DataFrame:
        """
        Load data from file based on extension.
        
        Args:
            file_path: Path to the data file
            **kwargs: Additional arguments passed to pandas reader
            
        Returns:
            pandas DataFrame
        """
        path = Path(file_path)
        
        if not path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        
        ext = path.suffix.lower()
        
        if ext not in DataLoader.SUPPORTED_FORMATS:
            raise ValueError(f"Unsupported format: {ext}. Supported: {DataLoader.SUPPORTED_FORMATS}")
        
        print(f"📂 Loading: {path.name}")
        
        if ext == '.csv':
            df = pd.read_csv(file_path, **kwargs)
        elif ext in ['.xlsx', '.xls']:
            df = pd.read_excel(file_path, **kwargs)
        elif ext == '.json':
            df = pd.read_json(file_path, **kwargs)
        elif ext == '.parquet':
            df = pd.read_parquet(file_path, **kwargs)
        
        print(f"✅ Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
        return df
    
    @staticmethod
    def load_multiple(file_paths: list) -> Dict[str, pd.DataFrame]:
        """
        Load multiple files into a dictionary of DataFrames.
        """
        datasets = {}
        for path in file_paths:
            name = Path(path).stem
            datasets[name] = DataLoader.load(path)
        return datasets

In [ ]:
# Test loading CSV files
titanic = DataLoader.load("../data/titanic.csv")
sales = DataLoader.load("../data/demo_sales.csv")

print("\n📁 Loaded datasets:")
print(f"   Titanic: {titanic.shape}")
print(f"   Sales: {sales.shape}")

## 2. Data Profiler

In [ ]:
class DataProfiler:
    """
    Generate comprehensive data profile for a DataFrame.
    """
    
    @staticmethod
    def profile(df: pd.DataFrame, name: str = "Dataset") -> Dict[str, Any]:
        """
        Generate a complete profile of the dataset.
        
        Returns:
            Dictionary with profile information
        """
        profile = {
            'name': name,
            'shape': {
                'rows': df.shape[0],
                'columns': df.shape[1]
            },
            'memory_usage': f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB",
            'columns': {},
            'missing_summary': {},
            'duplicates': {
                'count': df.duplicated().sum(),
                'percentage': f"{df.duplicated().mean() * 100:.2f}%"
            }
        }
        
        # Column-level analysis
        for col in df.columns:
            col_info = {
                'dtype': str(df[col].dtype),
                'non_null': int(df[col].notna().sum()),
                'null_count': int(df[col].isna().sum()),
                'null_pct': f"{df[col].isna().mean() * 100:.1f}%",
                'unique': int(df[col].nunique()),
            }
            
            # Add stats for numeric columns
            if df[col].dtype in ['int64', 'float64']:
                col_info['min'] = float(df[col].min()) if not pd.isna(df[col].min()) else None
                col_info['max'] = float(df[col].max()) if not pd.isna(df[col].max()) else None
                col_info['mean'] = float(df[col].mean()) if not pd.isna(df[col].mean()) else None
                col_info['std'] = float(df[col].std()) if not pd.isna(df[col].std()) else None
            
            # Add top values for categorical columns
            if df[col].dtype == 'object' or df[col].nunique() < 20:
                col_info['top_values'] = df[col].value_counts().head(5).to_dict()
            
            profile['columns'][col] = col_info
            
            # Track missing values
            if df[col].isna().sum() > 0:
                profile['missing_summary'][col] = col_info['null_pct']
        
        return profile
    
    @staticmethod
    def print_profile(profile: Dict[str, Any]):
        """
        Pretty print the profile.
        """
        print("\n" + "="*60)
        print(f"📊 DATA PROFILE: {profile['name']}")
        print("="*60)
        
        print(f"\n📐 Shape: {profile['shape']['rows']} rows × {profile['shape']['columns']} columns")
        print(f"💾 Memory: {profile['memory_usage']}")
        print(f"🔄 Duplicates: {profile['duplicates']['count']} ({profile['duplicates']['percentage']})")
        
        if profile['missing_summary']:
            print("\n⚠️  Missing Values:")
            for col, pct in profile['missing_summary'].items():
                print(f"   - {col}: {pct}")
        else:
            print("\n✅ No missing values!")
        
        print("\n📋 Column Types:")
        dtypes = {}
        for col, info in profile['columns'].items():
            dtype = info['dtype']
            dtypes[dtype] = dtypes.get(dtype, 0) + 1
        for dtype, count in dtypes.items():
            print(f"   - {dtype}: {count} columns")

In [ ]:
# Profile Titanic dataset
titanic_profile = DataProfiler.profile(titanic, "Titanic")
DataProfiler.print_profile(titanic_profile)

In [ ]:
# View detailed column info
print("\n📝 Column Details (first 5):")
for col, info in list(titanic_profile['columns'].items())[:5]:
    print(f"\n   {col}:")
    for key, value in info.items():
        if key != 'top_values':
            print(f"      {key}: {value}")

## 3. Data Quality Validator

In [ ]:
class DataValidator:
    """
    Validate data quality and check for common issues.
    """
    
    @staticmethod
    def validate(df: pd.DataFrame, rules: Optional[Dict] = None) -> Dict[str, Any]:
        """
        Run validation checks on DataFrame.
        
        Args:
            df: DataFrame to validate
            rules: Optional custom validation rules
            
        Returns:
            Validation report dictionary
        """
        issues = []
        warnings = []
        
        # Check 1: Empty DataFrame
        if df.empty:
            issues.append("DataFrame is empty")
            return {'status': 'FAILED', 'issues': issues, 'warnings': warnings}
        
        # Check 2: Missing values
        missing_pct = df.isna().mean()
        high_missing = missing_pct[missing_pct > 0.5].index.tolist()
        if high_missing:
            warnings.append(f"Columns with >50% missing: {high_missing}")
        
        # Check 3: Duplicate rows
        dup_pct = df.duplicated().mean()
        if dup_pct > 0.1:
            warnings.append(f"High duplicate rate: {dup_pct*100:.1f}%")
        
        # Check 4: Constant columns (no variance)
        constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
        if constant_cols:
            warnings.append(f"Constant/single-value columns: {constant_cols}")
        
        # Check 5: High cardinality columns
        high_cardinality = []
        for col in df.select_dtypes(include=['object']).columns:
            if df[col].nunique() > 0.9 * len(df):
                high_cardinality.append(col)
        if high_cardinality:
            warnings.append(f"High cardinality categorical columns: {high_cardinality}")
        
        # Check 6: Numeric columns with outliers (using IQR)
        outlier_cols = []
        for col in df.select_dtypes(include=[np.number]).columns:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
            if outliers > 0.05 * len(df):  # More than 5% outliers
                outlier_cols.append((col, outliers))
        if outlier_cols:
            warnings.append(f"Columns with significant outliers: {[c[0] for c in outlier_cols]}")
        
        # Custom rules validation
        if rules:
            for rule_name, rule_func in rules.items():
                try:
                    result = rule_func(df)
                    if not result:
                        issues.append(f"Custom rule failed: {rule_name}")
                except Exception as e:
                    issues.append(f"Error in rule '{rule_name}': {str(e)}")
        
        # Determine overall status
        if issues:
            status = 'FAILED'
        elif warnings:
            status = 'WARNING'
        else:
            status = 'PASSED'
        
        return {
            'status': status,
            'issues': issues,
            'warnings': warnings,
            'stats': {
                'rows': len(df),
                'columns': len(df.columns),
                'missing_cells': df.isna().sum().sum(),
                'duplicate_rows': df.duplicated().sum()
            }
        }
    
    @staticmethod
    def print_report(report: Dict[str, Any]):
        """
        Print validation report.
        """
        status_icons = {'PASSED': '✅', 'WARNING': '⚠️', 'FAILED': '❌'}
        
        print("\n" + "="*60)
        print(f"{status_icons[report['status']]} VALIDATION REPORT: {report['status']}")
        print("="*60)
        
        print(f"\n📊 Stats:")
        for key, value in report['stats'].items():
            print(f"   - {key}: {value}")
        
        if report['issues']:
            print("\n❌ Issues:")
            for issue in report['issues']:
                print(f"   - {issue}")
        
        if report['warnings']:
            print("\n⚠️  Warnings:")
            for warning in report['warnings']:
                print(f"   - {warning}")

In [ ]:
# Validate Titanic dataset
validation_report = DataValidator.validate(titanic)
DataValidator.print_report(validation_report)

In [ ]:
# Validate with custom rules
custom_rules = {
    'has_target': lambda df: 'Survived' in df.columns,
    'min_rows': lambda df: len(df) >= 100,
    'no_negative_age': lambda df: (df['Age'].dropna() >= 0).all()
}

report_with_rules = DataValidator.validate(titanic, rules=custom_rules)
DataValidator.print_report(report_with_rules)

## 4. Dataset Registry

In [ ]:
class DatasetRegistry:
    """
    Registry to manage multiple datasets with metadata.
    """
    
    def __init__(self):
        self.datasets: Dict[str, pd.DataFrame] = {}
        self.metadata: Dict[str, Dict] = {}
    
    def register(self, name: str, df: pd.DataFrame, metadata: Optional[Dict] = None):
        """
        Register a dataset with optional metadata.
        """
        self.datasets[name] = df
        self.metadata[name] = metadata or {}
        self.metadata[name]['shape'] = df.shape
        self.metadata[name]['columns'] = list(df.columns)
        print(f"✅ Registered: {name} ({df.shape[0]} × {df.shape[1]})")
    
    def get(self, name: str) -> pd.DataFrame:
        """
        Get a dataset by name.
        """
        if name not in self.datasets:
            raise KeyError(f"Dataset not found: {name}")
        return self.datasets[name]
    
    def list_datasets(self):
        """
        List all registered datasets.
        """
        print("\n📁 Registered Datasets:")
        for name, df in self.datasets.items():
            print(f"   - {name}: {df.shape[0]} rows × {df.shape[1]} columns")
    
    def get_info(self, name: str) -> Dict:
        """
        Get metadata for a dataset.
        """
        return self.metadata.get(name, {})

In [ ]:
# Create registry and register datasets
registry = DatasetRegistry()

registry.register(
    "titanic", 
    titanic,
    {'description': 'Titanic survival prediction', 'target': 'Survived', 'type': 'classification'}
)

registry.register(
    "sales",
    sales,
    {'description': 'Sales revenue prediction', 'target': 'Revenue', 'type': 'regression'}
)

registry.list_datasets()

In [ ]:
# Get dataset info
print("\n📝 Titanic Dataset Info:")
for key, value in registry.get_info('titanic').items():
    print(f"   {key}: {value}")

## ✅ Summary

This module provides:

**1. DataLoader**
- Load CSV, Excel, JSON, Parquet files
- Automatic format detection
- Batch loading multiple files

**2. DataProfiler**
- Comprehensive dataset profile
- Column-level statistics
- Missing value summary
- Memory usage tracking

**3. DataValidator**
- Automatic quality checks
- Custom validation rules
- Issue and warning reporting

**4. DatasetRegistry**
- Centralized dataset management
- Metadata tracking
- Easy dataset access